# Indoor Scene Change Detection — Training & Evaluation Pipeline


## 1. Environment Setup


In [ ]:
!nvidia-smi


In [ ]:
!pip install ultralytics pandas numpy scipy scikit-learn matplotlib seaborn opencv-python -q
import os, json, shutil, glob, zipfile, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from PIL import Image as PILImage
from sklearn.metrics import classification_report, confusion_matrix
from scipy.optimize import linear_sum_assignment
from ultralytics import YOLO
from google.colab import files
from IPython.display import Image, display

print("Libraries ready.")


## 10. Learned Classifier — Production Model


In [ ]:
from sklearn.ensemble import (RandomForestClassifier, HistGradientBoostingClassifier,
                               VotingClassifier)
from sklearn.model_selection import PredefinedSplit, RandomizedSearchCV
from sklearn.metrics import f1_score
from sklearn.base import clone

FEATURE_COLS_ALL = [
    'before_count', 'after_count', 'net_count_diff', 'total_objects',
    'max_iou', 'avg_iou', 'camera_shift_residual_px', 'move_thresh_used',
    'n_add', 'n_del', 'n_move',
    'best_add_conf', 'best_del_conf', 'best_move_conf', 'best_move_dist',
    'max_matched_dist', 'avg_appearance_sim', 'best_move_appearance_sim',
    'n_matched', 'frac_matched_before', 'frac_matched_after',
    'add_del_conf_margin', 'move_conf_margin', 'n_classes_changed', 'avg_matched_area_ratio',
    'total_events', 'move_event_ratio', 'is_net_ambiguous',
]

train_df = results_df[results_df['split'] == 'train']
val_df   = results_df[results_df['split'] == 'validation']
test_df  = results_df[results_df['split'] == 'test']
print(f"Train: {len(train_df)}   Validation: {len(val_df)}   Test: {len(test_df)}")

# --- Prune highly-correlated / redundant features (fit on TRAIN only) ---
corr_matrix = train_df[FEATURE_COLS_ALL].fillna(0.0).corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape, dtype=bool), k=1))
to_drop = set()
for col in upper.columns:
    if col in to_drop:
        continue
    hits = upper.index[upper[col] > 0.95].tolist()
    to_drop.update(hits)
FEATURE_COLS = [c for c in FEATURE_COLS_ALL if c not in to_drop]
print(f"Dropped {len(to_drop)} redundant features (|corr| > 0.95): {sorted(to_drop)}")
print(f"Using {len(FEATURE_COLS)}/{len(FEATURE_COLS_ALL)} features: {FEATURE_COLS}")

X_train, y_train = train_df[FEATURE_COLS].fillna(0.0), train_df['true_change']
X_val,   y_val   = val_df[FEATURE_COLS].fillna(0.0),   val_df['true_change']
X_test,  y_test  = test_df[FEATURE_COLS].fillna(0.0),  test_df['true_change']


X_search = pd.concat([X_train, X_val])
y_search = pd.concat([y_train, y_val])
test_fold = np.concatenate([np.full(len(X_train), -1), np.zeros(len(X_val))])
val_split = PredefinedSplit(test_fold)


rf_param_dist = {
    'n_estimators': [200, 300, 400, 500],
    'max_depth': [3, 4, 5, 6, 8],
    'min_samples_leaf': [3, 5, 8, 10, 15],
    'min_samples_split': [4, 6, 10, 15],
    'max_features': ['sqrt', 'log2', 0.5, 0.7],
}
rf_search = RandomizedSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=42),
    rf_param_dist, n_iter=60, cv=val_split, scoring='f1_macro',
    random_state=42, n_jobs=-1, refit=False)
rf_search.fit(X_search, y_search)
rf_best = RandomForestClassifier(class_weight='balanced', random_state=42, **rf_search.best_params_)
print("Best RandomForest params (selected on VALIDATION):", rf_search.best_params_)

hgb_param_dist = {
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.02, 0.03, 0.05, 0.08],
    'l2_regularization': [0.5, 1.0, 2.0, 5.0, 10.0],
    'max_leaf_nodes': [15, 31, 63],
}
hgb_search = RandomizedSearchCV(
    HistGradientBoostingClassifier(random_state=42, max_iter=1000,
                                    early_stopping=True, n_iter_no_change=20,
                                    validation_fraction=0.15),
    hgb_param_dist, n_iter=60, cv=val_split, scoring='f1_macro',
    random_state=42, n_jobs=-1, refit=False)
hgb_search.fit(X_search, y_search)
hgb_best = HistGradientBoostingClassifier(
    random_state=42, max_iter=1000, early_stopping=True, n_iter_no_change=20,
    validation_fraction=0.15, **hgb_search.best_params_)
print("Best HistGradientBoosting params (selected on VALIDATION):", hgb_search.best_params_)

voting = VotingClassifier(estimators=[('rf', clone(rf_best)), ('hgb', clone(hgb_best))], voting='soft')


candidates = {
    'RandomForest (tuned)': rf_best,
    'HistGradientBoosting (tuned)': hgb_best,
    'Voting Ensemble': voting,
}

selection_scores = {}
for name, clf in candidates.items():
    clf.fit(X_train, y_train)
    val_pred = clf.predict(X_val)
    f1 = f1_score(y_val, val_pred, average='macro')
    selection_scores[name] = f1
    print(f"{name}: validation macro-F1 = {f1:.3f}")

best_name = max(selection_scores, key=selection_scores.get)
best_arch = candidates[best_name]
print(f"\n-> Selected architecture: {best_name} (validation macro-F1 {selection_scores[best_name]:.3f})")

X_trainval = pd.concat([X_train, X_val])
y_trainval = pd.concat([y_train, y_val])
ml_clf_trainval = clone(best_arch)
ml_clf_trainval.fit(X_trainval, y_trainval)
test_pred = ml_clf_trainval.predict(X_test)

print(f"\n=== FINAL TEST-SET EVALUATION -- {best_name} (test touched exactly once) ===")
print(classification_report(y_test, test_pred))
test_report = classification_report(y_test, test_pred, output_dict=True)
print(f"Test accuracy: {test_report['accuracy']:.3f}   |   Test macro F1: {test_report['macro avg']['f1-score']:.3f}")

train_pred_check = ml_clf_trainval.predict(X_train)
train_f1_check = f1_score(y_train, train_pred_check, average='macro')
print(f"\nSanity check -- Train macro-F1: {train_f1_check:.3f}  |  Val macro-F1: {selection_scores[best_name]:.3f}  |  "
      f"Test macro-F1: {test_report['macro avg']['f1-score']:.3f}")
print("(Large train-vs-val/test gaps here signal overfitting -- re-check regularization ranges above if the gap reopens.)")

trainval_pred_all = ml_clf_trainval.predict(results_df[FEATURE_COLS].fillna(0.0))
results_df['predicted_change'] = trainval_pred_all
results_df['predicted_object'] = [
    object_for_type(pred, ctx['stats'], ctx['matched_pairs'])
    for pred, ctx in zip(trainval_pred_all, pair_contexts)
]

X_all = results_df[FEATURE_COLS].fillna(0.0)
y_all = results_df['true_change']
ml_clf = clone(best_arch)
ml_clf.fit(X_all, y_all)
print(f"\nProduction model ({best_name}) refit on all {len(results_df)} labeled pairs for deployment.")

if hasattr(ml_clf, 'feature_importances_'):
    importances = pd.Series(ml_clf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
    print("\nFeature importances (production model, which signals it actually relies on):")
    print(importances)
else:
    print("\n(Feature importances aren't directly exposed on this model type -- inspect "
          "rf_best.feature_importances_ separately if you need them.)")


### Final classifier confusion matrix (TEST split, evaluated once)


In [ ]:
labels_sorted_ml = sorted(set(y_test) | set(test_pred))
cm_ml = confusion_matrix(y_test, test_pred, labels=labels_sorted_ml)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_ml, annot=True, fmt='d', cmap='Greens',
            xticklabels=labels_sorted_ml, yticklabels=labels_sorted_ml)
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title(f'{best_name} -- Confusion Matrix (TEST split, n={len(y_test)})')
plt.tight_layout()
plt.savefig('/content/change_confusion_matrix_ml.png', dpi=150)
plt.show()


### Classification Report Heatmap (TEST split)


In [ ]:
report_rows = [l for l in labels_sorted_ml if l in test_report]
report_df_plot = pd.DataFrame(test_report).transpose().loc[report_rows, ['precision', 'recall', 'f1-score']]

plt.figure(figsize=(6, 4))
sns.heatmap(report_df_plot, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1)
plt.title(f'{best_name} -- Precision / Recall / F1 by Class (TEST split)')
plt.tight_layout()
plt.savefig('/content/classification_report_heatmap.png', dpi=150)
plt.show()


### ROC Curves (One-vs-Rest, TEST split)


In [ ]:
from sklearn.metrics import roc_curve, auc

if hasattr(ml_clf_trainval, 'predict_proba'):
    classes_sorted = list(ml_clf_trainval.classes_)
    y_score = ml_clf_trainval.predict_proba(X_test)

    plt.figure(figsize=(7, 6))
    for i, cls in enumerate(classes_sorted):
        y_true_bin = (y_test == cls).astype(int)
        if y_true_bin.sum() == 0 or y_true_bin.sum() == len(y_true_bin):
            continue  # class absent (or the only class) in TEST -- ROC undefined, skip
        fpr, tpr, _ = roc_curve(y_true_bin, y_score[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f'{cls} (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Chance')
    plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
    plt.title(f'{best_name} -- One-vs-Rest ROC Curves (TEST split)')
    plt.legend(loc='lower right')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('/content/classifier_roc_curves.png', dpi=150)
    plt.show()
else:
    print("This model type doesn't expose predict_proba -- skipping ROC curves.")


### Inspect Predictions


In [ ]:
print("True change distribution by split:")
print(results_df.groupby('split')['true_change'].value_counts().unstack(fill_value=0))
print()
print("Predicted change distribution by split (train/val predictions are optimistic --")
print("only 'test' is a fair check, see Section 10):")
print(results_df.groupby('split')['predicted_change'].value_counts().unstack(fill_value=0))
print()
print("If Add/Delete are still overrepresented on the TEST row, the likely cause is real")
print("detection instability near the threshold (a box flickering in/out of detection")
print("between frames). Try CONF_THRESH values further from the model's natural decision")
print("boundary (e.g. 0.5 or 0.7) and compare -- there's no universally 'more correct'")
print("value, it depends on where your model's false-positive/false-negative trade-off sits.")


In [ ]:
test_rows = results_df[results_df['split'] == 'test']
wrong = test_rows[test_rows['true_change'] != test_rows['predicted_change']]
print(f"{len(wrong)} / {len(test_rows)} TEST pairs misclassified")
wrong[['pair_id', 'true_change', 'predicted_change', 'true_object', 'predicted_object',
       'before_count', 'after_count', 'net_count_diff']].head(15)


### 10a. Overall System Performance Summary (TEST split only)


In [ ]:
active_test_metrics = yolo_test_metrics if ACTIVE_MODEL_PATH == YOLO_WEIGHTS else rtdetr_test_metrics
active_detector_name = active_test_metrics['model']

system_summary = pd.DataFrame([
    {'component': f'Detector\n({active_detector_name})', 'metric': 'mAP50',    'score': active_test_metrics['mAP50']},
    {'component': f'Detector\n({active_detector_name})', 'metric': 'mAP50-95', 'score': active_test_metrics['mAP50-95']},
    {'component': 'Change\nClassifier',                   'metric': 'Accuracy', 'score': test_report['accuracy']},
    {'component': 'Change\nClassifier',                   'metric': 'Macro F1', 'score': test_report['macro avg']['f1-score']},
])

colors_map = {'mAP50': '#4C72B0', 'mAP50-95': '#8CA9D6', 'Accuracy': '#DD8452', 'Macro F1': '#F0B27A'}

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(system_summary['component'] + '\n' + system_summary['metric'],
              system_summary['score'],
              color=[colors_map[m] for m in system_summary['metric']])
for b, v in zip(bars, system_summary['score']):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.01, f'{v:.2f}', ha='center', fontweight='bold')
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('Overall System Performance -- TEST Split Only (touched once)')
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('/content/system_performance_summary.png', dpi=150)
plt.show()

print("Every number in this chart comes from the TEST split, never validation --")
print("this is the figure to use for the report's overall-system-performance claim.")


## 11. Export for WEKA


In [ ]:
results_df.to_csv('/content/results_full_with_predictions.csv', index=False)

weka_df = results_df[FEATURE_COLS + ['true_change']].copy()
weka_df.to_csv('/content/weka_features.csv', index=False)

def df_to_arff(df, relation_name, arff_path, numeric_cols, class_col):
    with open(arff_path, 'w') as f:
        f.write(f"@RELATION {relation_name}\n\n")
        for c in numeric_cols:
            f.write(f"@ATTRIBUTE {c} NUMERIC\n")
        classes = sorted(df[class_col].unique())
        f.write(f"@ATTRIBUTE {class_col} {{{','.join(classes)}}}\n\n")
        f.write("@DATA\n")
        for _, r in df.iterrows():
            vals = [str(r[c]) for c in numeric_cols] + [str(r[class_col])]
            f.write(','.join(vals) + '\n')
    print(f"Wrote {arff_path}")

df_to_arff(weka_df, 'indoor_change_detection_features', '/content/weka_features.arff', FEATURE_COLS, 'true_change')
print("Saved: /content/weka_features.csv and /content/weka_features.arff (clean, no leakage, all splits combined)")

for split_name in ['train', 'validation', 'test']:
    split_df = results_df[results_df['split'] == split_name][FEATURE_COLS + ['true_change']].copy()
    split_df.to_csv(f'/content/weka_features_{split_name}.csv', index=False)
    df_to_arff(split_df, f'indoor_change_detection_{split_name}',
               f'/content/weka_features_{split_name}.arff', FEATURE_COLS, 'true_change')

print("Saved: /content/results_full_with_predictions.csv (full detail incl. split, for the report only)")


## 11b. Export for Flask Deployment


In [ ]:
import joblib

DEPLOY_DIR = '/content/deploy'
os.makedirs(DEPLOY_DIR, exist_ok=True)

shutil.copy(ACTIVE_MODEL_PATH, os.path.join(DEPLOY_DIR, 'best.pt'))

joblib.dump(ml_clf, os.path.join(DEPLOY_DIR, 'rf_classifier.joblib'))
print(f"Packaged classifier: {best_name}")

deploy_config = {
    'conf_thresh': CONF_THRESH,
    'move_min_thresh': MOVE_MIN_THRESH,
    'move_safety_factor': MOVE_SAFETY_FACTOR,
    'max_match_dist_frac': MAX_MATCH_DIST_FRAC,
    'min_appearance_sim': MIN_APPEARANCE_SIM,
    'clahe_clip_limit': CLAHE_CLIP_LIMIT,
    'clahe_tile_grid': list(CLAHE_TILE_GRID),
    'feature_cols': FEATURE_COLS,
    'class_names': list(active_model.names.values()),
}
with open(os.path.join(DEPLOY_DIR, 'config.json'), 'w') as f:
    json.dump(deploy_config, f, indent=2)

print("Deployment bundle contents:")
for fn in os.listdir(DEPLOY_DIR):
    print(' ', fn)
!cd /content && zip -rq flask_model_bundle.zip deploy
files.download('/content/flask_model_bundle.zip')
print("\nDownloaded flask_model_bundle.zip -- unzip its contents into flask_app/model/ "
      "(see the Flask app's README.md).")


## 12. Package & Download Everything


In [ ]:
!mkdir -p /content/results/models/yolo11s /content/results/models/rtdetr /content/results/charts
!cp {YOLO_WEIGHTS} /content/results/models/yolo11s/best.pt
!cp {RTDETR_WEIGHTS} /content/results/models/rtdetr/best.pt
for _png in [
    'dataset_distribution.png', 'dataset_distribution_by_split.png',
    'yolo11s_training_curves.png', 'rtdetr_training_curves.png',
    'model_comparison_test.png',
    'change_confusion_matrix_ml.png', 'classification_report_heatmap.png',
    'classifier_roc_curves.png',
    'system_performance_summary.png',
]:
    !cp /content/{_png} /content/results/charts/ 2>/dev/null || true
!cp /content/weka_features.csv /content/results/
!cp /content/weka_features.arff /content/results/
!cp /content/weka_features_train.csv /content/results/ 2>/dev/null || true
!cp /content/weka_features_train.arff /content/results/ 2>/dev/null || true
!cp /content/weka_features_validation.csv /content/results/ 2>/dev/null || true
!cp /content/weka_features_validation.arff /content/results/ 2>/dev/null || true
!cp /content/weka_features_test.csv /content/results/ 2>/dev/null || true
!cp /content/weka_features_test.arff /content/results/ 2>/dev/null || true
!cp /content/results_full_with_predictions.csv /content/results/
!cp -r /content/deploy /content/results/deploy 2>/dev/null || true
comparison_df.to_csv('/content/results/model_metrics_validation.csv', index=False)
comparison_test_df.to_csv('/content/results/model_metrics_test.csv', index=False)
!zip -rq /content/results.zip /content/results
files.download('/content/results.zip')
print("Downloaded results.zip")
